# DSAP — Validación del Parquet oficial de NYC 311

## Objetivo

Verificar la estructura, integridad y cobertura histórica del archivo
NYC 311 recuperado del almacenamiento S3 del equipo.

Comparar este archivo con el dataset utilizado previamente para el
entrenamiento de los modelos de clasificación de demanda semanal.

## Metodología

1. Identificar la procedencia de los archivos.
2. Validar la estructura e integridad de los datos.
3. Comparar registros y solicitudes.
4. Evaluar diferencias que puedan afectar el modelado.
5. Documentar la necesidad de un nuevo entrenamiento.

In [1]:
from pathlib import Path
import pandas as pd

raiz = Path.cwd()
if raiz.name == "notebooks":
    raiz = raiz.parent

ruta_oficial = raiz / "data/raw/nyc311_api_s3_nuevo.parquet"
ruta_anterior = raiz / "data/raw/nyc311_diario_borough_tipo_respaldo.parquet"

assert ruta_oficial.exists(), "No se encontró el archivo descargado de S3."
assert ruta_anterior.exists(), "No se encontró el archivo de respaldo."

df_oficial = pd.read_parquet(ruta_oficial)
df_anterior = pd.read_parquet(ruta_anterior)

print("Archivo S3:", ruta_oficial.name)
print("Dimensiones:", df_oficial.shape)
print("\nColumnas:", df_oficial.columns.tolist())

display(df_oficial.head())

Archivo S3: nyc311_api_s3_nuevo.parquet
Dimensiones: (652697, 4)

Columnas: ['borough', 'dia', 'complaint_type', 'n']


,borough,dia,complaint_type,n
0,BRONX,2022-01-01,Abandoned Vehicle,3
1,BRONX,2022-01-01,Animal-Abuse,2
2,BRONX,2022-01-01,Animal in a Park,1
3,BRONX,2022-01-01,APPLIANCE,5
4,BRONX,2022-01-01,Bike/Roller/Skate Chronic,1


## 4. Impacto de los cambios en la demanda semanal

Se determina si las diferencias entre los archivos modifican los
totales semanales utilizados para construir las variables predictoras
y la clasificación de demanda.

In [2]:
boroughs = [
    "BRONX", "BROOKLYN", "MANHATTAN",
    "QUEENS", "STATEN ISLAND"
]

def calcular_demanda_semanal(df):
    datos = df[df["borough"].isin(boroughs)].copy()

    datos["dia"] = pd.to_datetime(datos["dia"])
    datos["semana"] = (
        datos["dia"]
        - pd.to_timedelta(datos["dia"].dt.weekday, unit="D")
    ).dt.normalize()

    semanal = (
        datos.groupby(["borough", "semana"], as_index=False)["n"]
        .sum()
        .rename(columns={"n": "total_solicitudes"})
    )

    return semanal[
        semanal["semana"].between(
            "2022-01-03", "2025-12-22"
        )
    ].copy()


semanal_anterior = calcular_demanda_semanal(df_anterior)
semanal_s3 = calcular_demanda_semanal(df_oficial)

comparacion_semanal = semanal_anterior.merge(
    semanal_s3,
    on=["borough", "semana"],
    how="outer",
    suffixes=("_anterior", "_s3"),
    indicator=True
)

comparacion_semanal["diferencia"] = (
    comparacion_semanal["total_solicitudes_s3"].fillna(0)
    - comparacion_semanal["total_solicitudes_anterior"].fillna(0)
)

semanas_diferentes = comparacion_semanal[
    (comparacion_semanal["_merge"] != "both")
    | (comparacion_semanal["diferencia"] != 0)
].copy()

print("Semanas-borough con diferencias:", len(semanas_diferentes))
print("Diferencia neta de solicitudes:", semanas_diferentes["diferencia"].sum())

display(
    semanas_diferentes.sort_values(
        ["borough", "semana"]
    )
)

Semanas-borough con diferencias: 5
Diferencia neta de solicitudes: -11


,borough,semana,total_solicitudes_anterior,total_solicitudes_s3,_merge,diferencia
276,BROOKLYN,2023-04-24,19161,19164,both,3
414,BROOKLYN,2025-12-15,23542,23541,both,-1
622,MANHATTAN,2025-12-15,14723,14724,both,1
811,QUEENS,2025-08-04,19495,19482,both,-13
915,STATEN ISLAND,2023-08-07,2707,2706,both,-1


## 5. Evaluación del impacto sobre el modelado

La comparación de demanda semanal identificó cinco combinaciones de
borough y semana con diferencias entre las fuentes.

Aunque las diferencias absolutas son pequeñas, se debe comprobar su
impacto sobre las variables rezagadas, las medias móviles y la
clasificación de demanda antes de repetir el entrenamiento.

In [3]:
resumen_impacto = pd.DataFrame({
    "Indicador": [
        "Semanas-borough comparadas",
        "Semanas-borough con diferencias",
        "Porcentaje de semanas afectadas",
        "Diferencia neta de solicitudes",
        "Diferencia absoluta acumulada",
        "Mayor diferencia absoluta"
    ],
    "Resultado": [
        len(comparacion_semanal),
        len(semanas_diferentes),
        round(
            len(semanas_diferentes) / len(comparacion_semanal) * 100,
            4
        ),
        semanas_diferentes["diferencia"].sum(),
        semanas_diferentes["diferencia"].abs().sum(),
        semanas_diferentes["diferencia"].abs().max()
    ]
})

display(resumen_impacto)

,Indicador,Resultado
0,Semanas-borough comparadas,1040.0000
1,Semanas-borough con diferencias,5.0000
2,Porcentaje de semanas afectadas,0.4808
3,Diferencia neta de solicitudes,-11.0000
4,Diferencia absoluta acumulada,19.0000
5,Mayor diferencia absoluta,13.0000


## 6. Conclusiones de la validación

Se compararon los datos utilizados originalmente para el entrenamiento
con la nueva versión recuperada del almacenamiento S3 del equipo.

De las 1.040 combinaciones de borough y semana analizadas, únicamente
cinco presentaron diferencias, equivalentes al 0,4808 %.

La diferencia neta fue de -11 solicitudes y la diferencia absoluta
acumulada de 19 solicitudes.

Aunque las variaciones son pequeñas frente al volumen de información,
se realizará nuevamente la construcción del dataset y el entrenamiento
de los modelos utilizando el archivo recuperado de S3.

Esto permitirá mantener una fuente común de información para el equipo
y documentar cualquier variación en las métricas de evaluación.